# Case Study 2: Baseline DA Methods — Poor Reconstruction

This notebook demonstrates the degradation of classical DA methods under model mismatch:
- **Forcing:** Corrupted by Ornstein-Uhlenbeck process
- **Parameters:** Biased by 5% (σ=9.5, ρ=26.6, β=2.9)
- **Expected:** All methods degrade significantly (RMSE 3–6x higher than CS1)

In [ ]:
# --- Setup: Colab detection ---
import sys, os
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/rfablet/4dvarnet-fm-opencode.git
    %cd 4dvarnet-fm-opencode
    !pip install -r requirements.txt
else:
    sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from data.lorenz63 import Lorenz63Config, Lorenz63Dataset, generate_long_trajectory, generate_corrupted_forcing
from evaluation.baselines import Weak4DVar, Strong4DVar, EnKF

device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
# --- Parameters (edit these) ---
SEED = 123
DURATION = 5.0   # seconds
BIAS = 0.05      # 5% parameter bias
print(f'Seed: {SEED}, Duration: {DURATION}s, Bias: {BIAS*100:.0f}%')

## 1. Trajectory Divergence: True Model vs Noisy Model

Forward-simulate from the same initial condition to see how biased parameters and corrupted forcing cause the noisy model to drift away from the truth.

In [ ]:
div_cfg = Lorenz63Config(case=1, num_windows=1, T_max=10.0, seed=SEED)
n_steps = div_cfg.num_steps
dt = div_cfg.dt

sigma_b, rho_b, beta_b = Lorenz63Config(case=2, param_bias=BIAS).biased_params
print(f'True parameters:  sigma={div_cfg.sigma_true}, rho={div_cfg.rho_true}, beta={div_cfg.beta_true:.4f}')
print(f'Biased parameters: sigma={sigma_b:.2f}, rho={rho_b:.2f}, beta={beta_b:.4f}')

traj_true = generate_long_trajectory(
    n_steps, dt, SEED,
    div_cfg.sigma_true, div_cfg.rho_true, div_cfg.beta_true,
    div_cfg.gamma, div_cfg.W_L_bar, div_cfg.c1, div_cfg.c2,
    div_cfg.sigma_0, div_cfg.sigma_L, device,
)

traj_biased = generate_long_trajectory(
    n_steps, dt, SEED,
    sigma_b, rho_b, beta_b,
    div_cfg.gamma, div_cfg.W_L_bar, div_cfg.c1, div_cfg.c2,
    div_cfg.sigma_0, div_cfg.sigma_L, device,
)

W_L_true = traj_true[:, 3]
W_L_corrupted = generate_corrupted_forcing(W_L_true, n_steps, dt, div_cfg.tau_eta, div_cfg.sigma_eta, SEED + 2, device)
traj_true_state = traj_true[:, :3].numpy()
traj_biased_state = traj_biased[:, :3].numpy()
div_time = div_cfg.time_grid

rmse_div = np.sqrt(np.mean((traj_true_state - traj_biased_state)**2))
print(f'Final RMSE between trajectories: {rmse_div:.4f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for i, (ax, comp) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
    ax.plot(div_time, traj_true_state[:, i], color='blue', linewidth=1.5, label='True model', alpha=0.8)
    ax.plot(div_time, traj_biased_state[:, i], color='red', linewidth=1.5, linestyle='--', label='Noisy model', alpha=0.8)
    ax.fill_between(div_time, traj_true_state[:, i], traj_biased_state[:, i], alpha=0.15, color='purple')
    ax.set_ylabel(comp, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
axes[0].set_title('CS2: Trajectory Divergence — Noisy Model Drifts from Truth', fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(div_time, W_L_true.numpy(), color='green', linewidth=1.5, label='True Forcing W_L', alpha=0.8)
ax.plot(div_time, W_L_corrupted.numpy(), color='orange', linewidth=1.5, linestyle='--', label='Corrupted W_L* (OU)', alpha=0.8)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel(r'$W_L$', fontsize=12)
ax.set_title('CS2: True Forcing vs Corrupted Forcing', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## 2. Generate CS2 Dataset

In [ ]:
cfg = Lorenz63Config(case=2, param_bias=BIAS, num_windows=1, T_max=DURATION, seed=SEED)
dataset = Lorenz63Dataset(cfg)
window = dataset[0]

sigma, rho, beta = cfg.da_params
print(f'Case: 2 (corrupted forcing, biased parameters)')
print(f'Window: {cfg.num_steps} steps ({DURATION}s)')
print(f'DA parameters: sigma={sigma:.2f}, rho={rho:.2f}, beta={beta:.4f}')
print(f'Observations: {window["obs_mask"].sum().item()} sparse samples')

## 3. Run Baseline DA Methods

In [ ]:
true_state = window['true_state']
obs = window['obs']
obs_mask = window['obs_mask']
forcing = window['forcing_corrupted']  # CS2 uses corrupted forcing

weak_4dvar = Weak4DVar(da_window_steps=100, B_var=cfg.B_var, R_var=cfg.R_var,
                        opt_steps=150, dt=cfg.dt, device=device)
strong_4dvar = Strong4DVar(da_window_steps=100, B_var=cfg.B_var, R_var=cfg.R_var,
                            max_iter=40, dt=cfg.dt, device=device)
enkf = EnKF(N_ensemble=30, R_var=cfg.R_var, dt=cfg.dt, device=device)

print('Running Weak-4DVar...')
result_weak = weak_4dvar.assimilate(obs, obs_mask, forcing, true_state, sigma=sigma, rho=rho, beta=beta)
print('Running Strong-4DVar...')
result_strong = strong_4dvar.assimilate(obs, obs_mask, forcing, true_state, sigma=sigma, rho=rho, beta=beta)
print('Running EnKF...')
result_enkf = enkf.assimilate(obs, obs_mask, forcing, true_state, sigma=sigma, rho=rho, beta=beta)
print('Done.')

## 4. RMSE Table

In [ ]:
results = {'Weak-4DVar': result_weak, 'Strong-4DVar': result_strong, 'EnKF': result_enkf}

print('=' * 70)
print('  CASE STUDY 2: Noisy forcings & biased parameters')
print('=' * 70)
print(f"{'Method':<18} {'RMSE X':>12} {'RMSE Y':>12} {'RMSE Z':>12} {'Mean RMSE':>12}")
print('-' * 70)
for name, res in results.items():
    rmse = res.rmse
    mean_rmse = np.mean(rmse)
    print(f"{name:<18} {rmse[0]:>12.4f} {rmse[1]:>12.4f} {rmse[2]:>12.4f} {mean_rmse:>12.4f}")
print('=' * 70)
print('Expected: RMSE 3-6x higher than CS1 (significant degradation)')
print('=' * 70)

## 5. Reconstruction Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
true_state_np = true_state.numpy()
obs_np = obs.numpy()
obs_mask_np = obs_mask.numpy()
time_grid = cfg.time_grid

colors = {'Truth': 'black', 'Weak-4DVar': '#1f77b4', 'Strong-4DVar': '#ff7f0e', 'EnKF': '#2ca02c'}
for i, (ax, comp) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
    ax.plot(time_grid, true_state_np[:, i], color='black', linewidth=2, label='Truth', alpha=0.8)
    obs_times = time_grid[obs_mask_np]
    ax.scatter(obs_times, obs_np[obs_mask_np, i], color='gray', s=10, alpha=0.5, label='Obs', zorder=3)
    ax.plot(time_grid, result_weak.trajectory[:, i], color='#1f77b4', linewidth=1.5, linestyle='--', label='Weak-4DVar', alpha=0.8)
    ax.plot(time_grid, result_strong.trajectory[:, i], color='#ff7f0e', linewidth=1.5, linestyle='-.', label='Strong-4DVar', alpha=0.8)
    ax.plot(time_grid, result_enkf.trajectory[:, i], color='#2ca02c', linewidth=1.5, linestyle=':', label='EnKF', alpha=0.8)
    ax.set_xlabel('Time (s)', fontsize=11)
    ax.set_ylabel(comp, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', fontsize=9, framealpha=0.9)

fig.suptitle('CS2: Poor Reconstruction (corrupted forcing, biased parameters)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Forcing Impact Visualization

Shows how corrupted forcing leads to poor reconstruction quality.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
forcing_true = window['forcing_true'].numpy()
forcing_corrupted = window['forcing_corrupted'].numpy()

axes[0].plot(time_grid, forcing_true, color='green', linewidth=2, label='True Forcing', alpha=0.8)
axes[0].plot(time_grid, forcing_corrupted, color='orange', linewidth=2, linestyle='--', label='Corrupted Forcing (OU)', alpha=0.8)
axes[0].set_ylabel(r'$W_L$ (Forcing)', fontsize=12)
axes[0].set_title('CS2: Impact of Forcing Corruption on DA Performance', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=11, framealpha=0.9)
axes[0].grid(True, alpha=0.3, linestyle='--')
diff = forcing_corrupted - forcing_true
stats = f'Corruption Std: {np.std(diff):.4f}\nMean |Δ|: {np.mean(np.abs(diff)):.4f}'
axes[0].text(0.02, 0.98, stats, transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes[1].plot(time_grid, true_state_np[:, 0], color='black', linewidth=2, label='Truth (X)', alpha=0.8)
axes[1].plot(time_grid, result_strong.trajectory[:, 0], color='#ff7f0e', linewidth=1.5, linestyle='--', label='Strong-4DVar', alpha=0.8)
axes[1].fill_between(time_grid, true_state_np[:, 0], result_strong.trajectory[:, 0], alpha=0.3, color='red', label='Reconstruction Error')
axes[1].set_xlabel('Time (s)', fontsize=12)
axes[1].set_ylabel('X Component', fontsize=12)
axes[1].legend(loc='upper right', fontsize=11, framealpha=0.9)
axes[1].grid(True, alpha=0.3, linestyle='--')
rmse_x = result_strong.rmse[0]
axes[1].text(0.02, 0.02, f'RMSE (X): {rmse_x:.4f}', transform=axes[1].transAxes, fontsize=10,
             verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))
plt.tight_layout()
plt.show()

### Summary
- **CS2** demonstrates significant degradation of classical DA methods under model mismatch
- Biased parameters (5%) + corrupted forcing (OU process) cause all methods to fail
- RMSE values are 3–6x higher than CS1
- This validates the need for data-driven approaches like 4DVarNet-FM